# Semana 12: Apuntes de la clase

**Estructura temporal de tasas de interés** (CFA L2, Fixed Income: *The Term Structure and Interest Rate Dynamics*)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/12_curva_rendimientos/clase12_apuntes.ipynb)

Este notebook resume los conceptos de la sesión, con los ejemplos numéricos ejecutables. Hasta ahora usamos una sola tasa libre de riesgo; en el mercado hay una para cada plazo, y esa curva contiene las expectativas sobre la política monetaria. Esta semana no tiene práctica en Python ni tarea semanal. La sección 8 es la guía para leer los indicadores 5, 6 y 7 de la ficha de la PC2. Las celdas no requieren internet ni librerías adicionales.

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas y símbolos que usamos esta semana; los de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| $s_T$ | Tasa *spot* a $T$ años (*spot rate, zero rate*) | Rendimiento de un bono cupón cero que vence en $T$. |
| $DF_T$ | Factor de descuento (*discount factor*) | Precio hoy de una unidad monetaria pagadera en $T$. |
| YTM | *Yield to maturity* (rendimiento al vencimiento) | Tasa única que iguala el valor presente de los flujos de un bono a su precio. |
| Tasa par | *Par rate* | Cupón que hace que un bono cotice a su valor nominal. |
| $f_{a,b}$ | Tasa *forward* | Tasa pactada hoy para un préstamo que empieza en $a$ años y dura $b$ años. |
| *Bootstrapping* | Extracción sucesiva | Obtener la curva *spot* a partir de bonos con cupón, plazo por plazo. |
| Prima por plazo | *Term premium* | Compensación adicional por mantener un bono largo en lugar de renovar bonos cortos. |
| *Riding the yield curve* | Retorno por permanencia en la curva | Comprar un bono de plazo mayor que el horizonte y venderlo antes del vencimiento. |
| $D$ | Duración modificada (*modified duration*) | Cambio porcentual del precio ante un cambio de un punto en el rendimiento. |
| pb | Puntos básicos (*basis points, bp*) | Centésimas de punto porcentual: 50 pb = 0.50 por ciento. |
| Pendiente | *Slope* | Rendimiento a 10 años menos rendimiento a 2 años. |

## 1. Tasas spot y factores de descuento

La **estructura temporal** es la relación entre el plazo y la tasa de los instrumentos libres de riesgo de incumplimiento de un mismo soberano y una misma moneda. La tasa *spot* $s_T$ es el rendimiento de un cupón cero a $T$ años, y con ella se construye el factor de descuento:

$$DF_T = \frac{1}{(1 + s_T)^T} \qquad\qquad P_0 = \sum_{t=1}^{N} \frac{CF_t}{(1 + s_t)^t}$$

Cada flujo se descuenta con la tasa de **su propio** plazo. Curva del ejemplo: 4.0, 4.5 y 5.0 por ciento a 1, 2 y 3 años. Es ilustrativa: no es la curva de la que sale el $R_f = 4.5$ por ciento del curso.

In [ ]:
spot = {1: 0.040, 2: 0.045, 3: 0.050}
df = {t: 1 / (1 + s) ** t for t, s in spot.items()}

def precio(cupon, plazo, nominal=100):
    flujos = {t: cupon for t in range(1, plazo + 1)}
    flujos[plazo] += nominal
    return sum(cf * df[t] for t, cf in flujos.items()), flujos

p, flujos = precio(5, 3)
for t in df:
    print(f"Ano {t}: DF = {df[t]:.4f}  flujo = {flujos[t]:6.2f}  VP = {flujos[t] * df[t]:6.2f}")
print(f"Precio del bono a 3 anos con cupon de 5%: {p:.2f}")

Salida esperada: factores de descuento 0.9615, 0.9157 y 0.8638; valores presentes 4.81, 4.58 y 90.70; precio 100.09.

Ese es el precio **libre de arbitraje**: si el bono cotizara a otro precio, alguien podría comprarlo y vender sus flujos por separado (o lo contrario) con ganancia segura.

## 2. Rendimiento al vencimiento, tasa par y bootstrapping

El **YTM** es la tasa única que iguala el valor presente de los flujos al precio: un promedio complejo de las tasas *spot*, ponderado por los flujos. La **tasa par** es el cupón que hace que el bono cotice a 100:

$$c_N = \frac{1 - DF_N}{\sum_{t=1}^{N} DF_t}$$

El ***bootstrapping*** recorre el camino inverso: de bonos con cupón a la curva *spot*, plazo por plazo.

In [ ]:
def ytm(precio_bono, flujos, a=0.0, b=1.0):
    f = lambda y: sum(cf / (1 + y) ** t for t, cf in flujos.items()) - precio_bono
    for _ in range(200):                      # biseccion
        m = (a + b) / 2
        a, b = (m, b) if f(m) > 0 else (a, m)
    return (a + b) / 2

print(f"YTM del bono a 3 anos: {ytm(p, flujos):.2%}")

par = {n: (1 - df[n]) / sum(df[t] for t in range(1, n + 1)) for n in df}
for n, c in par.items():
    print(f"Tasa par a {n} ano(s): {c:.2%}   tasa spot: {spot[n]:.2%}")

# Bootstrapping: de la tasa par a 2 anos a la tasa spot a 2 anos
c2 = par[2] * 100
resto = 100 - c2 / (1 + spot[1])
s2 = ((100 + c2) / resto) ** 0.5 - 1
print(f"Bono par a 2 anos con cupon {c2:.2f}: VP del primer cupon {c2 / (1 + spot[1]):.2f}, resto {resto:.2f}, s2 = {s2:.2%}")

Salida esperada: YTM de 4.97 por ciento; tasas par de 4.00, 4.49 y 4.97 frente a tasas *spot* de 4.00, 4.50 y 5.00; el *bootstrapping* recupera $s_2 = 4.50$ por ciento.

Con pendiente positiva, la curva *spot* queda por encima de la curva par (y lo contrario con curva invertida), porque el bono con cupón recibe parte de sus flujos antes y esos flujos se descuentan a tasas menores. Las curvas que publican los bancos centrales suelen ser de rendimientos de bonos con cupón: para la ficha de la PC2 se usan tal como se publican.

## 3. Tasas forward

La tasa *forward* $f_{a,b}$ está implícita en la curva *spot* por ausencia de arbitraje:

$$(1 + s_{a+b})^{a+b} = (1 + s_a)^a\,(1 + f_{a,b})^b$$

In [ ]:
def forward(a, b):
    return ((1 + spot[a + b]) ** (a + b) / (1 + spot[a]) ** a) ** (1 / b) - 1

f11, f21 = forward(1, 1), forward(2, 1)
print(f"f(1,1) = {f11:.2%}   f(2,1) = {f21:.2%}")
print(f"Verificacion: (1.04)(1 + f11)(1 + f21) = {(1 + spot[1]) * (1 + f11) * (1 + f21):.4f} = (1.05)^3 = {1.05 ** 3:.4f}")
print(f"Item 1: {(1.04 ** 2) / 1.03 - 1:.2%}")

Salida esperada: $f_{1,1} = 5.00$ y $f_{2,1} = 6.01$ por ciento; el producto de las tres tasas de un periodo reproduce $(1.05)^3 = 1.1576$. El ítem 1 da 5.01.

Tres lecturas. Invertir a 2 años al 4.5 equivale a invertir a 1 año al 4.0 y reinvertir el segundo año al 5.0. La tasa *spot* es el promedio geométrico de las tasas *forward* de un periodo. Y con pendiente positiva las tasas *forward* quedan por encima de las *spot*: para subir el promedio, la tasa marginal tiene que ser mayor que el promedio.

## 4. Formas de la curva y teorías

| Teoría | Proposición | La tasa *forward* es |
|---|---|---|
| Expectativas puras | Los plazos son sustitutos perfectos | la tasa futura esperada |
| Preferencia por liquidez | Prima creciente con el plazo | la esperada más una prima positiva |
| Mercados segmentados | Oferta y demanda propias por plazo | ajena a las expectativas |
| Hábitat preferido | Se cambia de plazo por una prima | la esperada más una prima variable |

Las expectativas puras explican por qué las tasas de distintos plazos se mueven juntas, pero no por qué la pendiente es positiva la mayor parte del tiempo. La preferencia por liquidez sí: una curva puede tener pendiente positiva aunque nadie espere alzas de la tasa corta. La segmentación y el hábitat preferido explican distorsiones en plazos específicos (la demanda de bonos largos de fondos de pensiones y aseguradoras).

## 5. Lectura de la pendiente

$$\text{Pendiente} \approx \text{cambio esperado de la tasa corta} + \text{prima por plazo}$$

In [ ]:
prima_por_plazo = 0.004
print(f"Bajo expectativas puras, tasa a 1 ano esperada dentro de un ano: {f11:.2%}")
print(f"Con prima por plazo de {prima_por_plazo:.1%}: {f11 - prima_por_plazo:.2%} (alza esperada de {f11 - prima_por_plazo - spot[1]:.2%} y no de {f11 - spot[1]:.2%})")

def leer_pendiente(y2, y10, umbral_pb=25):
    pendiente = (y10 - y2) * 100               # en puntos basicos, con tasas en %
    if pendiente > umbral_pb:
        lectura = "positiva: tasa de politica baja respecto de la esperada; expectativa de alzas"
    elif pendiente < -umbral_pb:
        lectura = "negativa: politica contractiva y expectativa de recortes"
    else:
        lectura = "cercana a cero: tasa de politica proxima a su maximo esperado"
    return pendiente, lectura

for pais, y2, y10 in [("Pais A", 4.2, 5.6), ("Pais B", 5.1, 5.2), ("Pais C", 6.0, 5.3)]:
    pb, lectura = leer_pendiente(y2, y10)
    print(f"{pais}: {pb:+.0f} pb -> {lectura}")

Salida esperada: 5.00 por ciento bajo expectativas puras y 4.60 con prima por plazo (alza esperada de 0.60 y no de 1.00). País A: $+140$ pb, positiva. País B: $+10$ pb, cercana a cero. País C: $-70$ pb, negativa.

Los tres países son inventados; reemplaza los valores por los de tu ficha. El umbral de 25 puntos básicos es una convención del código, no una regla. Dos advertencias: la pendiente indica dirección esperada, no fecha ni magnitud; y la prima por plazo no es observable, así que nunca sabemos con exactitud cuánto de la pendiente es expectativa.

## 6. Retorno por permanencia en la curva

Con pendiente positiva y una curva que **no cambia**, un bono de plazo mayor que el horizonte rinde más que el bono de plazo igual al horizonte.

In [ ]:
p0 = 1 / (1 + spot[2]) ** 2                  # cupon cero a 2 anos, hoy
for tasa_futura in (0.040, 0.050, 0.060):
    p1 = 1 / (1 + tasa_futura)               # dentro de un ano le queda 1 ano
    print(f"Tasa a 1 ano dentro de un ano = {tasa_futura:.1%}: retorno de B = {p1 / p0 - 1:.2%} frente a {spot[1]:.2%} de A")

Salida esperada: 5.00 por ciento si la curva no cambia (la tasa a 1 año sigue en 4.0); 4.00 si la tasa futura resulta igual a la *forward* (5.0), lo mismo que la alternativa A; 3.02 si sube a 6.0.

El punto de equilibrio es la tasa *forward*: la estrategia gana cuando las tasas futuras resultan **menores** que las *forward* y pierde cuando resultan mayores. Toda apuesta activa en renta fija es una opinión sobre si las tasas futuras serán distintas de las tasas *forward*.

## 7. Riesgo de tasa de interés

$$\frac{\Delta P}{P} \approx -D \times \Delta y$$

In [ ]:
y = ytm(p, flujos)
macaulay = sum(t * cf / (1 + y) ** t for t, cf in flujos.items()) / p
d_mod = macaulay / (1 + y)
print(f"Bono a 3 anos: duracion de Macaulay {macaulay:.2f}, duracion modificada {d_mod:.2f}")

alza = 0.005
aprox = -d_mod * alza
exacto = sum(cf / (1 + y + alza) ** t for t, cf in flujos.items()) / p - 1
print(f"Alza de 50 pb: aproximacion {aprox:.2%}, exacto {exacto:.2%}")
print(f"Bono con duracion 8 y alza de 50 pb: {-8 * alza:.1%}")

Salida esperada: duración de Macaulay 2.86 y modificada 2.72; un alza de 50 pb reduce el precio 1.36 por ciento según la duración y 1.35 por ciento con el cálculo exacto. El bono de duración 8 pierde 4.0 por ciento.

La duración aumenta con el plazo y disminuye con el cupón, y supone un desplazamiento **paralelo** de la curva. Los movimientos reales se descomponen en tres factores: **nivel** (todas las tasas en la misma dirección; explica la mayor parte de la variación), **pendiente** (cortas y largas en magnitudes distintas) y **curvatura** (el tramo medio respecto de los extremos).

## 8. La curva en la valorización y en la ficha de la PC2

- **$R_f$ del CAPM:** se toma del plazo consistente con la duración de los flujos. Para una empresa en marcha, el rendimiento soberano a 10 años, no la tasa de política.
- **Moneda:** cada moneda tiene su curva; la curva en soles incorpora la inflación esperada en soles.
- **Diferencial soberano:** el rendimiento de un bono soberano en dólares de un país emergente menos el del Tesoro de Estados Unidos del mismo plazo mide el riesgo país (semana 14).
- **Acciones:** un alza del tramo largo eleva $K_e$ y reduce más el valor de las empresas cuyos flujos están lejos en el tiempo.

**Cuatro preguntas para los indicadores 5, 6 y 7 de la ficha:**

1. **Nivel.** ¿El rendimiento a 2 años está por encima o por debajo de la tasa de referencia? Por debajo: el mercado espera recortes. Por encima: espera alzas.
2. **Pendiente.** ¿Positiva, cercana a cero o negativa? Aplica la guía de la sección 5.
3. **Coherencia.** ¿La lectura de la curva coincide con la fase del ciclo y la postura monetaria de tu informe?
4. **Comparación.** ¿Qué países tienen las curvas más empinadas? ¿La diferencia de nivel a 10 años entre países viene de la inflación esperada, de la tasa real o del riesgo?

## 9. Errores comunes

1. Descontar todos los flujos de un bono con una sola tasa *spot*, o con el YTM de otro bono.
2. Confundir la tasa *forward* con un pronóstico: es una tasa de equilibrio de arbitraje que incluye primas.
3. Concluir que una pendiente positiva implica alzas esperadas de la tasa corta.
4. Leer una curva invertida como una predicción con fecha de recesión.
5. Comparar un rendimiento a 10 años en soles con uno en dólares sin considerar la inflación esperada.
6. Usar la tasa de política como $R_f$ para valorar flujos de largo plazo.
7. Aplicar la duración a movimientos no paralelos o a cambios grandes de tasa.

## 10. Ítems resueltos

**Ítem 1.** $(1.04)^2/1.03 - 1 = 5.01$ por ciento. **Clave A.** 3.5 promedia las dos tasas *spot*; 7.0 las suma.

**Ítem 2.** Con preferencia por liquidez, la prima por plazo basta para dar pendiente positiva aunque la tasa corta esperada sea constante. **Clave B.** La opción A corresponde a expectativas puras; la C, a mercados segmentados.

In [ ]:
spot_item = {1: 0.05, 2: 0.06}
p_item = 6 / (1 + spot_item[1]) + 106 / (1 + spot_item[2]) ** 2
print(f"Item 3: {p_item:.2f}  (con 6% para ambos flujos: {6 / 1.06 + 106 / 1.06 ** 2:.2f}; con 5%: {6 / 1.05 + 106 / 1.05 ** 2:.2f})")

**Ítem 3.** $6/1.05 + 106/(1.06)^2 = 5.71 + 94.34 = 100.05$. **Clave C.** 100.00 descuenta ambos flujos al 6 por ciento; 101.86 descuenta ambos al 5.

**Ítem 4.** Si la curva no cambia, el cupón cero a 2 años vendido al cabo de un año rinde la tasa *forward* $f_{1,1}$. **Clave B.** La tasa *spot* a 1 año es el retorno del bono cuyo plazo coincide con el horizonte; la tasa *spot* a 2 años es el retorno anual si se mantiene hasta el vencimiento.

## 11. Lista de verificación

- Puedo valorar un bono con cupón descontando cada flujo con su tasa *spot*.
- Puedo calcular una tasa *forward* a partir de dos tasas *spot* y explicar qué significa.
- Puedo distinguir tasa *spot*, tasa par y YTM.
- Puedo enunciar las cuatro teorías y decir qué explica cada una.
- Puedo leer la pendiente de una curva real y relacionarla con el ciclo y la política monetaria.
- Puedo calcular el retorno de mantener un bono si la curva no cambia y decir cuándo la estrategia pierde.
- Puedo estimar el efecto de un cambio de tasa sobre el precio con la duración.

La semana 13 trata la política monetaria: instrumentos, mecanismos de transmisión y efecto sobre la curva y los precios de los activos.